In [12]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import pandas as pd
import itertools

# ------------------------
# PARAMETERS
# ------------------------
WIDTH = 380
HEIGHT = 280
NUM_VOTERS = 1000
NUM_PARTIES = 2
NUM_DISTRICTS = 8
VOTERS_PER_DISTRICT = NUM_VOTERS // NUM_DISTRICTS
NUM_RUNS = 10

NUM_CITIES = 8
CITY_INTENSITY = (10, 30)
CITY_SPREAD = (20, 60)

np.random.seed(1)

PARTY_IDS = [f"Party {i}" for i in range(NUM_PARTIES)]
party_colors = np.random.rand(NUM_PARTIES, 3)
PARTY_COLORS = dict(zip(PARTY_IDS, party_colors))

# ------------------------
# PARTY DISTRIBUTION (bias)
# ------------------------
PARTY_BIAS = [0.45, 0.55]

# ------------------------
# GENERATE POPULATION DENSITY
# ------------------------
density = np.full((HEIGHT, WIDTH), 1.0)
X, Y = np.meshgrid(np.arange(WIDTH), np.arange(HEIGHT))

for _ in range(NUM_CITIES):
    cx = np.random.uniform(0, WIDTH)
    cy = np.random.uniform(0, HEIGHT)
    intensity = np.random.uniform(*CITY_INTENSITY)
    sigma = np.random.uniform(*CITY_SPREAD)
    density += intensity * np.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * sigma**2))

# ------------------------
# GENERATE VOTERS BASED ON DENSITY
# ------------------------
flat_density = density.ravel()
flat_density /= flat_density.sum()
indices = np.random.choice(WIDTH * HEIGHT, size=NUM_VOTERS, p=flat_density)
voters_y, voters_x = np.unravel_index(indices, (HEIGHT, WIDTH))
voters = np.column_stack((voters_x, voters_y))

# ------------------------
# FIXED PARTY PREFERENCES
# ------------------------
voter_parties = np.random.choice(
    np.arange(NUM_PARTIES),
    size=NUM_VOTERS,
    p=PARTY_BIAS
)
voter_colors = party_colors[voter_parties]

# ------------------------
# CONNECTED DISTRICTS FUNCTION (BIASABLE)
# ------------------------
def connected_districts(
    voters,
    num_districts,
    voters_per_district,
    rng,
    voter_parties,
    target_party,
    k_neighbors=100,
    bias_strength=0.85,
    max_attempts=10
):
    N = len(voters)

    for attempt in range(max_attempts):
        district = -np.ones(N, dtype=int)
        tree = cKDTree(voters)
        unassigned = set(range(N))
        success = True

        for d in range(num_districts):
            if not unassigned:
                break

            seed = list(unassigned)[rng.integers(len(unassigned))]
            district[seed] = d
            unassigned.remove(seed)

            queue = [seed]
            count = 1
            counts = {p: 0 for p in range(len(PARTY_IDS))}
            counts[voter_parties[seed]] += 1

            while queue and count < voters_per_district:
                current = queue.pop(0)
                _, neighbors = tree.query(voters[current], k=k_neighbors)
                neighbors = np.atleast_1d(neighbors)
                neighbors = [n for n in neighbors if n in unassigned]
                if not neighbors:
                    continue

                def score(n):
                    party = voter_parties[n]
                    others = [counts[p] for p in counts if p != target_party]
                    max_other = max(others) if others else 0

                    if party == target_party:
                        if counts[target_party] > max_other + 2:
                            return 1
                        else:
                            return 3
                    else:
                        if counts[party] + 1 > counts[target_party]:
                            return -2
                        else:
                            return 0

                if rng.random() < bias_strength:
                    neighbors = sorted(neighbors, key=score, reverse=True)
                else:
                    neighbors = rng.permutation(neighbors)

                for n in neighbors:
                    if n in unassigned:
                        district[n] = d
                        unassigned.remove(n)
                        queue.append(n)
                        counts[voter_parties[n]] += 1
                        count += 1
                        if count >= voters_per_district:
                            break

            if count < voters_per_district:
                success = False
                break

        if success:
            # Assign leftovers
            leftover = np.where(district == -1)[0]
            assigned = np.where(district != -1)[0]
            if len(leftover) > 0:
                assigned_tree = cKDTree(voters[assigned])
                for i in leftover:
                    _, idx = assigned_tree.query(voters[i])
                    district[i] = district[assigned[idx]]
            return district

    raise RuntimeError("Failed to construct districts after multiple attempts")

# ------------------------
# COUNT VOTES
# ------------------------
def count_votes(district_labels):
    district_counts = []
    for d in range(NUM_DISTRICTS):
        mask = district_labels == d
        counts = {p: 0 for p in range(NUM_PARTIES)}
        for i in np.where(mask)[0]:
            counts[voter_parties[i]] += 1
        district_counts.append(counts)
    return district_counts

# ------------------------
# EFFICIENCY GAP
# ------------------------
def efficiency_gap(district_counts):
    district_gaps = []
    for district in district_counts:
        winning_party = max(district.items(), key=lambda item: item[1])[0]
        num_winning_votes = district[winning_party]
        total_votes = sum(district.values())
        num_losing_votes = total_votes - num_winning_votes
        wasted_winner = num_winning_votes - (total_votes / NUM_PARTIES)
        wasted_loser = num_losing_votes
        gap = (wasted_winner - wasted_loser) / total_votes
        district_gaps.append(gap)
    return district_gaps

# ------------------------
# WIN CHECK
# ------------------------
def get_is_winner(district_counts, target_party):
    target_votes = district_counts[target_party]
    other_votes = [district_counts[p] for p in district_counts if p != target_party]
    return all(target_votes > v for v in other_votes)

# ------------------------
# STEP CURVE FUNCTIONS
# ------------------------
def get_current_vote_share(district_counts, target_party):
    total = sum(sum(d.values()) for d in district_counts)
    party_total = sum(d[target_party] for d in district_counts)
    return party_total / total

def compute_step_curve_for_party(district_counts, target_party, PARTY_IDS, NUM_DISTRICTS):
    initial_vote = get_current_vote_share(district_counts, target_party)
    initial_wins = sum(1 for d in district_counts if get_is_winner(d, target_party))
    # Forward/backward simulations
    fwd = simulate_vote_shifts_forward(district_counts, target_party, PARTY_IDS, NUM_DISTRICTS)
    rev = simulate_vote_shifts_backward(district_counts, target_party, PARTY_IDS)
    base_point = (initial_vote, initial_wins)
    fwd_points = [base_point] + [(h["vote_share"], h["districts_won"]) for h in fwd["history"]]
    rev_points = [base_point] + [(h["vote_share"], h["districts_won"]) for h in rev["history"]]
    merged = sorted(set(fwd_points + rev_points))
    return merged

def plot_step_and_complement(step_points, NUM_DISTRICTS, title="Seat–Vote Curve"):
    x, y = zip(*sorted(step_points))
    x, y = np.array(x), np.array(y)
    y = y / NUM_DISTRICTS
    plt.figure(figsize=(8,6))
    plt.step(x, y, where="post", color="black", lw=2, label="Seat–Vote Curve")
    xc = 1 - x
    yc = 1 - y
    order = np.argsort(xc)
    plt.step(xc[order], yc[order], where="post", color="blue", linestyle="--", label="Complement Curve")
    plt.xlabel("Statewide Vote Share")
    plt.ylabel("Seat Share")
    plt.title(title)
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.show()

def compute_step_function_area(step_points, num_districts):
    points = sorted(step_points, key=lambda x: x[0])
    if points[0][0] > 0:
        points = [(0,0)] + points
    if points[-1][0] < 1:
        points = points + [(1.0, points[-1][1])]
    step_x, step_y = zip(*points)
    step_x = np.array(step_x)
    step_y = np.array(step_y) / num_districts
    complement_x = 1 - step_x[::-1]
    complement_y = 1 - step_y[::-1]
    all_x = np.unique(np.concatenate([step_x, complement_x]))
    area_between = 0.0
    area_step = 0.0
    area_complement = 0.0
    def step_value_at(x, xs, ys):
        idx = np.searchsorted(xs, x, side='right') - 1
        idx = max(0, idx)
        return ys[idx]
    for i in range(len(all_x)-1):
        x1 = all_x[i]
        x2 = all_x[i+1]
        width = x2 - x1
        midpoint = (x1 + x2)/2
        f_val = step_value_at(midpoint, step_x, step_y)
        g_val = step_value_at(midpoint, complement_x, complement_y)
        area_between += width * abs(f_val - g_val)
        area_step += width * f_val
        area_complement += width * g_val
    return {"area_between": area_between, "area_step": area_step, "area_complement": area_complement}

# ------------------------
# SIMULATION LOOP
# ------------------------
all_efficiency_gaps = []
all_pr_vote_shares = []
all_pr_seat_shares = []
all_district_counts = []
all_step_curves = []
all_area_results = []

for run in range(NUM_RUNS):
    rng = np.random.default_rng(run)
    TARGET_PARTY = 0
    district_labels = connected_districts(
        voters,
        NUM_DISTRICTS,
        VOTERS_PER_DISTRICT,
        rng,
        voter_parties=voter_parties,
        target_party=TARGET_PARTY,
        k_neighbors=50,
        bias_strength=0.85
    )

    district_counts = count_votes(district_labels)
    all_efficiency_gaps.append(efficiency_gap(district_counts))

    pr_vote_share = {p:0 for p in range(NUM_PARTIES)}
    pr_seat_share = {p:0 for p in range(NUM_PARTIES)}

    for d in district_counts:
        for p in d:
            pr_vote_share[p] += d[p]
        winner = max(d, key=d.get)
        pr_seat_share[winner] += 1

    # Normalize
    total_votes = sum(pr_vote_share.values())
    for p in pr_vote_share:
        pr_vote_share[p] /= total_votes
    for p in pr_seat_share:
        pr_seat_share[p] /= NUM_DISTRICTS

    all_pr_vote_shares.append(pr_vote_share)
    all_pr_seat_shares.append(pr_seat_share)
    all_district_counts.append(district_counts)

    # Step curves
    step_curves_this_run = {}
    area_results_this_run = {}
    for p in range(NUM_PARTIES):
        curve = compute_step_curve_for_party(district_counts, p, PARTY_IDS, NUM_DISTRICTS)
        step_curves_this_run[p] = curve
        area_results_this_run[p] = compute_step_function_area(curve, NUM_DISTRICTS)

    all_step_curves.append(step_curves_this_run)
    all_area_results.append(area_results_this_run)

# ------------------------
# PLOT VOTE VS SEAT
# ------------------------
plt.figure(figsize=(8,8))
for p in range(NUM_PARTIES):
    vote_shares = [run[p] for run in all_pr_vote_shares]
    seat_shares = [run[p] for run in all_pr_seat_shares]
    plt.scatter(vote_shares, seat_shares, alpha=0.7, label=PARTY_IDS[p])

plt.plot([0,1],[0,1], linestyle='--', color='black', label='Perfect Proportionality')
plt.xlabel("Vote Share")
plt.ylabel("Seat Share")
plt.title("Seat Share vs Vote Share per Party")
plt.xlim(0,1)
plt.ylim(0,1)
plt.gca().set_aspect('equal', adjustable='box')
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

NameError: name 'simulate_vote_shifts_forward' is not defined

In [ ]:
# ------------------------------------------------------------
# WASTED VOTE RATIOS PER PARTY PER RUN
# ------------------------------------------------------------

wasted_vote_table = []

for run_idx in range(NUM_RUNS):

    district_counts = all_district_counts[run_idx]

    # Track wasted votes per party
    wasted_votes = {party: 0 for party in PARTY_IDS}
    total_votes = 0

    for district in district_counts:

        district_total = sum(district.values())
        total_votes += district_total

        # Winner of district
        winner = max(district, key=district.get)

        for party in PARTY_IDS:

            votes = district[party]

            if party == winner:
                wasted = votes - (district_total / NUM_PARTIES)
            else:
                wasted = votes

            wasted_votes[party] += wasted

    # Convert to ratios
    ratios = {party: wasted_votes[party] / total_votes for party in PARTY_IDS}

    row = {"Run": run_idx + 1}
    row.update(ratios)

    wasted_vote_table.append(row)

df_wasted = pd.DataFrame(wasted_vote_table)

print("\n--- WASTED VOTE RATIO (WASTED / TOTAL VOTES) ---")
print(df_wasted.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- MEAN WASTED VOTE RATIO ACROSS RUNS ---")
print(df_wasted.drop(columns="Run").mean())



# ------------------------------------------------------------
# EXPECTED WASTED VOTE RATIOS (ANALYTIC)
# ------------------------------------------------------------

# Compute total votes per party across all districts
total_votes_per_party = {party: 0 for party in PARTY_IDS}
total_votes_all = 0

# First, compute overall vote fractions
for district in all_district_counts[0]:  # use first run as representative
    district_total = sum(district.values())
    total_votes_all += district_total
    for party in PARTY_IDS:
        total_votes_per_party[party] += district[party]

# Compute vote shares
vote_shares = {party: total_votes_per_party[party] / total_votes_all for party in PARTY_IDS}

# Expected wasted vote ratios
expected_wasted = {}
for party in PARTY_IDS:
    p = vote_shares[party]
    expected_wasted[party] = (p * NUM_PARTIES - 1) / NUM_PARTIES  # approximate per-district formula

print("\n--- EXPECTED WASTED VOTE RATIO (ANALYTIC) ---")
for party, val in expected_wasted.items():
    print(f"Party {party}: {val:.4f}")


--- WASTED VOTE RATIO (WASTED / TOTAL VOTES) ---
 Run  Party 0  Party 1
   1   0.4180   0.0820
   2   0.4180   0.0820
   3   0.4180   0.0820
   4   0.4180   0.0820
   5   0.3555   0.1445
   6   0.4180   0.0820
   7   0.4180   0.0820
   8   0.4180   0.0820
   9   0.3555   0.1445
  10   0.4180   0.0820

--- MEAN WASTED VOTE RATIO ACROSS RUNS ---
Party 0    0.4055
Party 1    0.0945
dtype: float64

--- EXPECTED WASTED VOTE RATIO (ANALYTIC) ---
Party Party 0: -0.0820
Party Party 1: 0.0820


In [ ]:
print(np.bincount(district_labels))

[125 125 125 125 125 125 125 125]


In [ ]:
# ------------------------
# VISUALIZE ONE RUN (NO OVERLAP)
# ------------------------
rng = np.random.default_rng(42)  # Fixed seed for reproducibility
TARGET_PARTY = 0

district_labels = connected_districts(
    voters,
    NUM_DISTRICTS,
    VOTERS_PER_DISTRICT,
    rng,
    voter_parties=voter_parties,
    target_party=TARGET_PARTY,
    k_neighbors=50,
    bias_strength=0.85
)

# Build KD-tree for fast nearest lookup
tree = cKDTree(voters)

# Create grid of every pixel
xx, yy = np.meshgrid(np.arange(WIDTH), np.arange(HEIGHT))
grid_points = np.column_stack((xx.ravel(), yy.ravel()))

# Assign each pixel to nearest voter
_, nearest_voter = tree.query(grid_points)
pixel_districts = district_labels[nearest_voter]
pixel_map = pixel_districts.reshape((HEIGHT, WIDTH))

plt.figure(figsize=(10,7))

# Lightly color districts
plt.imshow(pixel_map, origin='lower', cmap='tab20', alpha=0.35)

# Draw district boundaries
plt.contour(
    pixel_map,
    levels=np.arange(NUM_DISTRICTS)+0.5,
    colors='black',
    linewidths=1,
    origin='lower'
)

# Plot voters on top
for i, party in enumerate(PARTY_IDS):
    mask = voter_parties == i
    plt.scatter(
        voters[mask, 0],
        voters[mask, 1],
        color=PARTY_COLORS[party],
        label=party,
        s=20
    )

plt.title("Voters and Districts (True Boundary Lines)")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.legend()
plt.show()

RuntimeError: Failed to construct districts after multiple attempts

In [ ]:
print("Run", run)
print([max(d, key=d.get) for d in district_counts])

Run 9
['Party 1', 'Party 1', 'Party 1', 'Party 1', 'Party 1', 'Party 1', 'Party 1', 'Party 1']
